# DocuExtract — Fine-Tuning LayoutLMv3 on SROIE in Google Colab
### End-to-end receipt key information extraction (Company, Date, Address, Total)

This notebook runs on a free **Google Colab T4 GPU** to fine-tune `microsoft/layoutlmv3-base` on the ICDAR 2019 SROIE dataset, evaluate per-field F1 scores, and export the trained model weights for local FastAPI deployment.

## 1. Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q transformers[torch] datasets seqeval evaluate torchvision pillow

## 2. Load Dataset from Hugging Face

In [ ]:
from datasets import load_dataset

# ICDAR 2019 SROIE mirror
dataset = load_dataset('jsdnrs/ICDAR2019-SROIE')
print(dataset)
print('Sample entry:', dataset['train'][0].keys())

## 3. Define Labels & BIO Schema

In [ ]:
label_list = [
    'O',
    'B-COMPANY', 'I-COMPANY',
    'B-DATE', 'I-DATE',
    'B-ADDRESS', 'I-ADDRESS',
    'B-TOTAL', 'I-TOTAL'
]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}
print('Labels:', label2id)

## 4. Initialize LayoutLMv3 Processor & Preprocessing Pipeline

In [ ]:
from transformers import AutoProcessor

model_id = 'microsoft/layoutlmv3-base'
processor = AutoProcessor.from_pretrained(model_id, apply_ocr=False)

def prepare_examples(examples):
    images = examples['image']
    words = examples['words']
    boxes = examples['bboxes']
    word_labels = examples['ner_tags']

    encoding = processor(
        images,
        words,
        boxes=boxes,
        word_labels=word_labels,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )
    return encoding

## 5. Load Model & Training Setup

In [ ]:
import torch
import evaluate
import numpy as np
from transformers import LayoutLMv3ForTokenClassification, TrainingArguments, Trainer

model = LayoutLMv3ForTokenClassification.from_pretrained(
    model_id,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

seqeval = evaluate.load('seqeval')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p_idx] for (p_idx, l_idx) in zip(prediction, label) if l_idx != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l_idx] for (p_idx, l_idx) in zip(prediction, label) if l_idx != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'overall_f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

training_args = TrainingArguments(
    output_dir='checkpoints_layoutlmv3',
    max_steps=1000,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    evaluation_strategy='steps',
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='overall_f1',
    fp16=True,
    logging_steps=50
)

## 6. Train the Model

In [ ]:
# Preprocess splits
encoded_dataset = dataset.map(
    prepare_examples,
    batched=True,
    batch_size=2,
    remove_columns=dataset['train'].column_names
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['test'],
    tokenizer=processor,
    compute_metrics=compute_metrics
)

trainer.train()

## 7. Evaluate & Inspect Classification Report

In [ ]:
eval_metrics = trainer.evaluate()
print('Final Evaluation Metrics:', eval_metrics)

## 8. Export Model Artifacts & Download

In [ ]:
import os
from google.colab import files

save_dir = 'best_layoutlmv3_sroie'
trainer.save_model(save_dir)
processor.save_pretrained(save_dir)

!zip -r best_layoutlmv3_sroie.zip best_layoutlmv3_sroie
print('Ready! Downloading model zip...')
files.download('best_layoutlmv3_sroie.zip')